# Задача 2. Permutations

На вход дан массив. Необходимо вернуть все возможные перестановки.

Примеры:

- `nums = [1, 2, 3]` → `[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]`
- `nums = [0, 1]` → `[[0, 1], [1, 0]]`
- `nums = [1]` → `[[1]]`

Обязательно предусмотреть визуализацию стека вызовов, в идеале использовать декоратор из первой задачи. Написать тесты.

## Идея

Строим перестановку по одному элементу. `path` — уже выбранные числа, `used[i]` показывает, взят ли элемент с индексом `i`. Рекурсивная функция пробует каждый свободный элемент, вызывает себя, затем убирает этот элемент, чтобы попробовать другой вариант. Когда длина `path` равна длине входа, копируем готовую перестановку в ответ.

Перед поиском сортируем копию массива. Если одинаковые значения стоят рядом, пропускаем второе на том же шаге, чтобы не возвращать одинаковые перестановки. Исходный массив при этом не меняется. Декоратор `tracer` из задачи 1 печатает вход и выход каждого рекурсивного вызова вместе с текущим `path`.

## Реализация

Этот ноутбук самостоятельный, поэтому декоратор из первой задачи повторён здесь.

In [1]:
def tracer(function):
    depth = 0

    def wrapper(*args, **kwargs):
        nonlocal depth
        parts = [repr(value) for value in args]
        parts += [f"{name}={value!r}" for name, value in kwargs.items()]
        arguments = ", ".join(parts)
        indent = "  " * depth

        print(f"{indent}→ {function.__name__}({arguments})")
        depth += 1
        try:
            result = function(*args, **kwargs)
        finally:
            depth -= 1
        print(f"{indent}← {function.__name__} = {result!r}")
        return result

    return wrapper

In [2]:
def permutations(nums):
    values = sorted(nums)
    used = [False] * len(values)
    result = []

    @tracer
    def backtrack(path):
        if len(path) == len(values):
            result.append(path.copy())
            return

        for i in range(len(values)):
            if used[i]:
                continue
            if i > 0 and values[i] == values[i - 1] and not used[i - 1]:
                continue

            used[i] = True
            path.append(values[i])
            backtrack(path)
            path.pop()
            used[i] = False

    backtrack([])
    return result

## Пример работы

Для `[1, 2, 3]` сначала выбираем `1`, затем `2`, затем `3`: получаем `[1, 2, 3]`. После возврата убираем `3`, пробуем другую ветку `[1, 3, 2]`. Когда все ветки с начальным `1` закончились, возвращаемся к пустому `path` и пробуем начать с `2`, затем с `3`.

| Глубина | `path` при входе | Что происходит |
|---:|:---|:---|
| 0 | `[]` | выбираем первое число |
| 1 | `[1]` | выбираем второе число |
| 2 | `[1, 2]` | выбираем третье число |
| 3 | `[1, 2, 3]` | копируем в ответ и возвращаемся |
| 2 | `[1, 2]` | убираем `3`, других вариантов здесь нет |
| 1 | `[1]` | убираем `2`, пробуем `3` |

Ниже та же работа видна в выводе декоратора: отступ показывает глубину стека.

In [3]:
answer = permutations([1, 2, 3])
print("Ответ:", answer)

→ backtrack([])
  → backtrack([1])
    → backtrack([1, 2])
      → backtrack([1, 2, 3])
      ← backtrack = None
    ← backtrack = None
    → backtrack([1, 3])
      → backtrack([1, 3, 2])
      ← backtrack = None
    ← backtrack = None
  ← backtrack = None
  → backtrack([2])
    → backtrack([2, 1])
      → backtrack([2, 1, 3])
      ← backtrack = None
    ← backtrack = None
    → backtrack([2, 3])
      → backtrack([2, 3, 1])
      ← backtrack = None
    ← backtrack = None
  ← backtrack = None
  → backtrack([3])
    → backtrack([3, 1])
      → backtrack([3, 1, 2])
      ← backtrack = None
    ← backtrack = None
    → backtrack([3, 2])
      → backtrack([3, 2, 1])
      ← backtrack = None
    ← backtrack = None
  ← backtrack = None
← backtrack = None
Ответ: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


## Сложность

Для `n` разных элементов существует `n!` перестановок. Каждую готовую перестановку копируем за `O(n)`, поэтому время без учёта печати трассы — `O(n · n!)`. Сортировка `O(n log n)` не меняет итоговую оценку. Ответ хранит `O(n · n!)` элементов; путь, массив `used` и стек рекурсии занимают ещё `O(n)`. Печать трассы требует дополнительного времени пропорционально объёму напечатанного текста.

## Тесты

Проверяем примеры из условия, пустой массив, повторяющиеся и отрицательные элементы, отсутствие изменения входного массива и количество вариантов для четырёх разных чисел.

In [4]:
from contextlib import redirect_stdout
from io import StringIO

def quiet_permutations(nums):
    with redirect_stdout(StringIO()):
        return permutations(nums)

assert quiet_permutations([1, 2, 3]) == [
    [1, 2, 3], [1, 3, 2], [2, 1, 3],
    [2, 3, 1], [3, 1, 2], [3, 2, 1]
]
assert quiet_permutations([0, 1]) == [[0, 1], [1, 0]]
assert quiet_permutations([1]) == [[1]]
assert quiet_permutations([]) == [[]]
assert quiet_permutations([1, 1, 2]) == [
    [1, 1, 2], [1, 2, 1], [2, 1, 1]
]
assert quiet_permutations([-1, 0]) == [[-1, 0], [0, -1]]

original = [3, 1, 2]
assert len(quiet_permutations(original)) == 6
assert original == [3, 1, 2]
answer = quiet_permutations([1, 2, 3, 4])
assert len(answer) == 24
assert len({tuple(item) for item in answer}) == 24

print("Задача 2: все тесты пройдены")

Задача 2: все тесты пройдены
